In [7]:
import keras
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from keras import layers

import numpy as np

In [2]:
#Load dataset
(x_train_tmp, y_train_tmp), (x_test_tmp, y_test_tmp) = keras.datasets.cifar100.load_data()
# print(y_train_tmp.shape)
x_train = []
y_train = []
x_test = []
y_test = []
for i in range (y_train_tmp.shape[0]):
    if (0 <= y_train_tmp[i][0] <= 19):
        x_train.append(x_train_tmp[i])
        y_train.append(y_train_tmp[i])
for i in range (y_test_tmp.shape[0]):
    if (0 <= y_test_tmp[i][0] <= 19):
        x_test.append(x_test_tmp[i])
        y_test.append(y_test_tmp[i])
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

x_train = x_train / 255.0
x_test = x_test / 255.0


trainY = to_categorical(y_train, num_classes = 20)
testY = to_categorical(y_test, num_classes = 20)

In [6]:
vgg16_model = keras.applications.VGG16(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=None,
    pooling=None,
)

# xception_model.summary()

model = keras.Sequential(
    [
        keras.Input(shape=(32, 32, 3)),
        vgg16_model,

        layers.Flatten(),

        layers.Dropout(0.4),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        
        layers.Dropout(0.4),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation("relu"),

        layers.Dense(20, activation='softmax')
    ]
)
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, None, None, 512)   14714688  
                                                                 
 flatten_1 (Flatten)         (None, 512)               0         
                                                                 
 dropout_2 (Dropout)         (None, 512)               0         
                                                                 
 dense_3 (Dense)             (None, 512)               262656    
                                                                 
 batch_normalization_2 (Bat  (None, 512)               2048      
 chNormalization)                                                
                                                                 
 activation_2 (Activation)   (None, 512)               0         
                                                      

In [8]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

model.fit(x_train, trainY, epochs=5, callbacks=[early_stopping], batch_size=32, validation_split=0.1)

Epoch 1/5


2025-05-24 19:26:56.724031: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 110592000 exceeds 10% of free system memory.


282/282 [==============================] - 350s 1s/step - loss: 2.9577 - accuracy: 0.0967 - val_loss: 2.9957 - val_accuracy: 0.0880
Epoch 2/5
282/282 [==============================] - 356s 1s/step - loss: 2.6936 - accuracy: 0.1379 - val_loss: 2.7466 - val_accuracy: 0.1400
Epoch 3/5
282/282 [==============================] - 339s 1s/step - loss: 2.5828 - accuracy: 0.1497 - val_loss: 2.7551 - val_accuracy: 0.1280
Epoch 4/5
282/282 [==============================] - 340s 1s/step - loss: 2.4978 - accuracy: 0.1664 - val_loss: 2.6944 - val_accuracy: 0.1220
Epoch 5/5
282/282 [==============================] - 335s 1s/step - loss: 2.4335 - accuracy: 0.1760 - val_loss: 2.5995 - val_accuracy: 0.1260


In [5]:
model.evaluate(x_test, testY)

63/63 [==============================] - 9s 147ms/step - loss: 2.5572 - accuracy: 0.1700


[2.557166576385498, 0.17000000178813934]